# 03 - Hyperparameter tuning

Three search methods plus a set of hand-picked parameters were tested against the dataset from
notebook 02. None beat XGBoost defaults.

| Configuration | Method | CV | MAE (sd) |
|---|---|---|---|
| Defaults | none | 10 x 3 | **10.77 (0.53)** |
| Optuna best | 100-trial TPE study | 5 x 5 | 11.34 (0.43) |
| Optuna best | 100-trial TPE study | 5 x 2 | 11.36 (0.46) |
| RandomizedSearchCV best | 25 iterations | 10 x 3 | 14.09 (0.64) |
| GridSearchCV best | 2,592 combinations | 10 x 3 | 15.03 (0.60) |
| Manual | hand-set | 10 x 3 | 15.03 to 16.15 |

The results below were recorded from separate runs. Code cells are left unexecuted so the
notebook reads in a sensible order rather than the order I actually ran things in.

Each search below optimises its own objective, then the winning parameters are scored with the
same `RepeatedKFold` protocol used in notebook 02 so the numbers can be compared.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna

from sklearn.model_selection import cross_val_score, RepeatedKFold, train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_excel(r'data/dataset.xlsx')
X = df.drop(columns=['price', 'no'])
y = df['price']

cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

def score(model, cv=cv):
    s = np.absolute(cross_val_score(model, X, y, scoring='neg_mean_absolute_error',
                                    cv=cv, n_jobs=-1, error_score='raise'))
    print('Mean MAE: %.3f (%.3f)' % (s.mean(), s.std()))
    return s.mean()

## Baseline

Stock parameters, the same run as notebook 02.

```
Mean MAE: 10.769 (0.531)
```

In [ ]:
score(xgb.XGBRegressor())

## 1. GridSearchCV

Exhaustive search over eight parameters. 3 x 4 x 3 x 2 x 2 x 1 x 3 x 3 x 2 = 2,592 combinations,
scored on negative mean squared error.

In [ ]:
parameters_for_testing = {
    'colsample_bytree': [0.4, 0.6, 0.8],
    'gamma': [0, 0.03, 0.1, 0.3],
    'min_child_weight': [1.5, 6, 10],
    'learning_rate': [0.1, 0.07],
    'max_depth': [3, 5],
    'n_estimators': [10000],
    'reg_alpha': [1e-5, 1e-2, 0.75],
    'reg_lambda': [1e-5, 1e-2, 0.45],
    'subsample': [0.6, 0.95],
}

xgb_model = xgb.XGBRegressor(learning_rate=0.1, n_estimators=1000, max_depth=5,
                             min_child_weight=1, gamma=0, subsample=0.8,
                             colsample_bytree=0.8, nthread=6, seed=27)

gsearch = GridSearchCV(estimator=xgb_model, param_grid=parameters_for_testing,
                       n_jobs=6, verbose=10, scoring='neg_mean_squared_error')
gsearch.fit(X, y)
print('best params:', gsearch.best_params_)
print('best score:', gsearch.best_score_)

Winning configuration, scored on the shared protocol:

```
Mean MAE: 15.032 (0.598)
```

Worse than defaults.

In [ ]:
score(xgb.XGBRegressor(colsample_bytree=0.4,
                       gamma=0,
                       learning_rate=0.07,
                       max_depth=5,
                       min_child_weight=1.5,
                       n_estimators=1000,
                       reg_alpha=0.75,
                       reg_lambda=0.45,
                       subsample=0.6,
                       seed=42))

## 2. RandomizedSearchCV

25 sampled configurations over six parameters, scored on negative mean absolute error.

In [ ]:
params = {
    'max_depth': [3, 5, 6, 10, 15, 20],
    'learning_rate': [0.01, 0.1, 0.2, 0.3],
    'subsample': np.arange(0.5, 1.0, 0.1),
    'colsample_bytree': np.arange(0.4, 1.0, 0.1),
    'colsample_bylevel': np.arange(0.4, 1.0, 0.1),
    'n_estimators': [100, 500, 1000],
}

clf = RandomizedSearchCV(estimator=xgb.XGBRegressor(seed=20),
                         param_distributions=params,
                         scoring='neg_mean_absolute_error',
                         n_iter=25, verbose=1)
clf.fit(X, y)
print('Best parameters:', clf.best_params_)

Best parameters found: `subsample` 0.7, `n_estimators` 1000, `max_depth` 10,
`learning_rate` 0.01, `colsample_bytree` 0.6, `colsample_bylevel` 0.8.

```
Mean MAE: 14.085 (0.636)
```

Better than the grid search, still worse than defaults.

In [ ]:
score(xgb.XGBRegressor(subsample=0.7,
                       n_estimators=1000,
                       max_depth=10,
                       learning_rate=0.01,
                       colsample_bytree=0.6,
                       colsample_bylevel=0.8))

## 3. Optuna

100 trials with a TPE sampler. The objective minimises RMSE on a held-out 15% split, with
100-round early stopping inside each trial, so `n_estimators` can be set high and the early stop
decides how long training actually runs.

| Parameter | Range |
|---|---|
| `lambda`, `alpha` | log-uniform, 1e-3 to 10.0 |
| `colsample_bytree` | 0.3 to 1.0 |
| `subsample` | 0.4 to 1.0 |
| `learning_rate` | 0.008 to 0.02 |
| `max_depth` | 5, 7, 9, 11, 13, 15, 17 |
| `min_child_weight` | 1 to 300 |

In [ ]:
def objective(trial, data=X, target=y):
    train_x, test_x, train_y, test_y = train_test_split(
        data, target, test_size=0.15, random_state=42)

    param = {
        'lambda': trial.suggest_loguniform('lambda', 1e-3, 10.0),
        'alpha': trial.suggest_loguniform('alpha', 1e-3, 10.0),
        'colsample_bytree': trial.suggest_categorical(
            'colsample_bytree', [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]),
        'subsample': trial.suggest_categorical(
            'subsample', [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]),
        'learning_rate': trial.suggest_categorical(
            'learning_rate', [0.008, 0.01, 0.012, 0.014, 0.016, 0.018, 0.02]),
        'n_estimators': 10000,
        'max_depth': trial.suggest_categorical('max_depth', [5, 7, 9, 11, 13, 15, 17]),
        'random_state': trial.suggest_categorical('random_state', [2020]),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 300),
    }

    model = xgb.XGBRegressor(**param)
    model.fit(train_x, train_y, eval_set=[(test_x, test_y)],
              early_stopping_rounds=100, verbose=False)
    preds = model.predict(test_x)
    return mean_squared_error(test_y, preds, squared=False)

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

In [ ]:
study.trials_dataframe()

Best trial:

```
{'lambda': 0.0356, 'alpha': 0.0046, 'colsample_bytree': 0.8, 'subsample': 0.6,
 'learning_rate': 0.014, 'max_depth': 13, 'min_child_weight': 10}
```

Scored on the shared protocol at two cross-validation settings:

```
5 splits x 5 repeats -> Mean MAE: 11.343 (0.428)
5 splits x 2 repeats -> Mean MAE: 11.356 (0.456)
```

Closest of the three, still short of the baseline.

In [ ]:
best = xgb.XGBRegressor(reg_lambda=0.0356,
                        reg_alpha=0.0046,
                        colsample_bytree=0.8,
                        subsample=0.6,
                        learning_rate=0.014,
                        max_depth=13,
                        min_child_weight=10,
                        n_estimators=10000,
                        random_state=2020)

score(best, cv=RepeatedKFold(n_splits=5, n_repeats=5, random_state=1))

## Conclusion

A 2,592-point exhaustive grid, a 25-iteration random search and a 100-trial Bayesian study, and
none of them improved on stock parameters. Searching that much of the parameter space without
moving the score points at the features rather than the estimator.

About 14% mean absolute error is what these features support.

Two directions from here. The listings carry free-text descriptions that this project does not
use. Their quality varies widely, from detailed write-ups to near-empty boilerplate, so the signal
is uneven and extracting it needs NLP work that was out of scope. It is still the most obvious
untapped source, since sellers often describe renovation, view and finish quality in prose while
leaving structured fields blank. Beyond that, the remaining ceiling is information the portal does
not hold at all, above all transaction prices rather than asking prices.

### Caveat

The comparison is not fully controlled. Cross-validation settings differ between runs (10 x 3 for
the grid and random searches, 5 x 5 and 5 x 2 for Optuna), and the Optuna objective was RMSE on a
single held-out split while the comparison metric is cross-validated MAE. The gaps are large
enough that the conclusion holds, but a clean re-run under one protocol would be needed to treat
these as benchmark numbers.